# CS-781 Phase 2 — Legal NER, round 2 (notebook F)

Same pipeline as notebook E (`phase2-ner`), plus three things that need **no retraining** and one training option:

| New | What it does | Where |
|---|---|---|
| **Viterbi decoding** | Today each token takes its own argmax tag, so one wrong tag in the middle of a long PREC/STAT span splits it into two wrong spans. Viterbi picks the best tag sequence over the whole document among the *valid BIO* ones (no `I-X` after `O` or after another label). It uses the model's probabilities as they are and only forbids invalid transitions; nothing is learned. | `decode: "viterbi"` (N3), lab |
| **Document-level propagation** | In the training gold, every case-sensitive occurrence of an annotated string inside a document has the same label (unless it sits inside a longer entity). Check it yourself: for the 70 training documents not one exception. So when the model tags a string at ≥ `min_frac` of its occurrences, every occurrence is tagged, with the string's majority label (Krishnan & Manning 2006). | `propagate` (N3), lab |
| **Ensembles** | Average the tag probabilities of several finished runs (all share the same tokens), or vote on their spans. Optional greedy forward selection on validation (Caruana et al. 2004). | lab (N11) |
| **Train on all 70 documents** | `train_on_all: True` + `epochs_from: "<a validation run>"` reuses that run's best epoch. Also `seed` per experiment. | N3, Part C |

Every new prediction set is scored on the **same 14 validation documents** as notebook E and written as a Codabench zip;
`submissions/index.csv` lists them all with validation strict macro-F1 / P / R. **You choose what to upload** — and every
setting marked YOUR CHOICE is yours (course rule §17.5). Declare in your LLM declaration that the pipeline code
(decoding, propagation, ensembling, lab) was written with an assistant.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · secret `Capstone_Phase_1` on.

**Inputs (Add Input):**
| Input | Needed for |
|---|---|
| `notebook857159535b` (data prep) | tokenizer — always |
| your Phase-1 checkpoints (same inputs as notebook E: notebook D, `Phase1_Training1`, dataset `prefix`, `phase1-train-extra`) | Part B (rebuild the old runs) and Part C (training) |
| **NEW: the output of notebook E `phase2-ner`** — the version(s) whose output has `ner_runs/<experiment>/best.pt`, `prediction.json`, `val_prediction.json` | Parts A and B. `swa_bitfit` was re-run in a later version: add that version too (or make a dataset from its output). N7 prints which folder each run comes from. |

**How to use**
1. **Session 1 (nothing to train):** leave `SESSION_EXPERIMENTS = []` in N3 → **Save & Run All** (~20–30 min).
   Part A post-processes the old span predictions, Part B rebuilds each old run from `best.pt` and saves its token
   scores, the lab (N11) decodes / propagates / ensembles them. Download `/kaggle/working/submissions/`
   (`index.csv` + the zips) from the Output tab.
2. **Later sessions:** add your own experiments in N3 and list them in `SESSION_EXPERIMENTS` (run the CPU smoke test
   first, as before). They train in Part C and flow into the lab automatically. Add this notebook's own output as an
   input next time, so token scores computed earlier are reused instead of recomputed.
3. Nothing here changes your N6 code (`ner_methods.py`): the head, LoRA/LoRMA/BitFit and distillation are yours, unchanged.

## N1. Login + download the Phase 2 data

In [ ]:
import os, shutil, json
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p2data", exist_ok=True)
for path in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json", "Train/fold2-gold.json",
             "Train/labels.txt", "Test/fold3-text.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p2data/{os.path.basename(path)}")
print(sorted(os.listdir("/kaggle/working/p2data")))

## N2. GPU check

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — your experiments (unchanged), the new options, and the lab settings

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
#  Write down why you chose them: the report must compare methods and justify settings.
# =====================================================================================================
#
#  PHASE 2 RULES THIS CONFIG MUST SATISFY (§8)
#  * all FOUR required Phase-1 architectures used: decoder, prefix (hybrid), encdec, swa (efficient)
#  * the best four submissions come from DIFFERENT Phase-1 models
#  * at least 3 different fine-tuning / parameter-efficient techniques — LORMA is COMPULSORY
#  * at least 1 knowledge-distillation experiment (bigger teacher -> smaller student) — COMPULSORY
#  * ≥ 6 submissions in total, each a different METHOD (not just other learning rates / ranks)
#
#  The experiments below are the ones you ran in notebook E (unchanged), so their names still match the
#  checkpoints of that run. Add new ones at the end.

CONFIG = {
    "seed": 42,

    # ---------------- validation split (same for EVERY experiment, so comparisons are fair) ----------------
    # Do not change these two: the lab compares new runs with the notebook-E runs on the SAME 14 documents.
    "n_val_docs": 14,
    "val_fraction": 0.2,

    # ---------------- where each Phase-1 model's checkpoint is (searched under /kaggle/input) ----------------
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",   # notebook D: decoder after continued training (val loss 2.656)
        "prefix":    "prefix/best.pt",                # the dataset you upload (Phase1_Training1 version 1)
        "encdec":    "encdec/best.pt",                # Phase1_Training1 latest version (session 2)
        "swa":       "swa/best.pt",
        "diffusion": "diffusion/best.pt",             # phase1-train-extra (session 3)
        "moe":       "moe/best.pt",
    },

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        "window": 1024,      # tokens per window (automatically capped at what the model accepts: encdec 768).
        "stride": 768,       # the next window starts this many tokens later -> 256 tokens of overlap.
        "context": "plain",  # "plain" or "echo" (causal models only: each window is fed twice and the second copy
                             # is read, so every token has seen the whole window — Springer et al. 2024).
        "epochs": 10,        # passes over the training windows; the best epoch (validation F1) is kept.
        "patience": 3,       # stop early after this many epochs without a better validation F1.
        "lr": 5e-5,          # YOUR CHOICE.
        "batch_size": 8,
        "warmup_frac": 0.1,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "head_dropout": 0.1,
        "class_weights": "none",   # "none" or "inv_sqrt" (rare tags cost more: weight ∝ 1/sqrt(count))
        "decode": "lenient", # how token tags become spans — also used to pick the best epoch:
                             #   "lenient": argmax per token; an I-X without a preceding B-X still starts an entity
                             #   "strict":  argmax per token; such an orphan I-X is ignored
                             #   "viterbi": NEW — the best tag sequence over the whole document among those that
                             #              are valid BIO (no I-X after O or after another label). Uses the model's
                             #              probabilities, forbids only invalid transitions; nothing is learned.
        "propagate": None,   # NEW. None = off, or a number min_frac in (0, 1]: document-level label propagation.
                             #   In the training gold, every case-sensitive occurrence of an annotated string in a
                             #   document carries the same label (unless it lies inside a longer entity). With
                             #   propagation, a string the model tagged in at least min_frac of its occurrences
                             #   is tagged everywhere in that document, with its majority label. Low min_frac =
                             #   more recall, but a wrong string gets copied too. The lab (N11) scores several values.
        "align": "token",    # "token": every token gets a tag; "piece": only the first token of each piece
                             # (a run of letters/digits, or one punctuation mark) — spans end on piece boundaries.
        "method_cfg": {},
        "moe_aux_coef": 0.0,
        "train_on_all": False,     # True = train on all 70 documents (no validation; fixed number of epochs)
        "epochs_from": None,       # NEW, for train_on_all runs: name of a finished VALIDATION run; its best epoch
                                   # becomes this run's number of epochs (instead of "epochs").
        "teacher": None,
        "kd": {"temperature": 2.0, "alpha": 0.5},
        "student_layers": None,
        # "seed": NEW — an experiment may set its own seed (default: the global seed above). Same settings with
        #         different seeds give different models, which the lab can average (ensemble).
    },

    # ---------------- the experiments ----------------
    "experiments": {
        # --- notebook E (already run; kept so their names and teachers resolve) ---
        "prefix_full":    {"arch": "prefix", "method": "full"},
        "encdec_lorma":   {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "decoder_lora":   {"arch": "decoder", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "swa_bitfit":     {"arch": "swa", "method": "bitfit", "lr": 1e-3},
        "diffusion_full": {"arch": "diffusion", "method": "full"},
        "moe_full":       {"arch": "moe", "method": "full"},
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full", "student_layers": 4},

        # --- YOUR NEW EXPERIMENTS (YOUR CHOICE) — syntax only, fill in your own values: ---
        # "name": {"arch": ..., "method": ..., <any default above you want to change>},
        # "name_all": {<the settings of a validated run>, "train_on_all": True, "epochs_from": "<that run>"},
        # "name_s2": {<the settings of a validated run>, "seed": <another integer>},
    },
}

# ---------------- which experiments THIS session trains (GPU), and which the CPU smoke test runs ----------------
SESSION_EXPERIMENTS = []          # empty = train nothing: the session only post-processes / re-decodes earlier runs
SMOKE_EXPERIMENTS = list(SESSION_EXPERIMENTS)   # a distillation student's teacher is smoke-tested first automatically

# ---------------- v2 lab: post-processing + ensembles of finished runs (N7, N9, N11) — YOUR CHOICE ----------------
PROP_MIN_FRACS = [0.1, 0.2, 0.3, 0.5]  # propagation thresholds the lab tries; each = one more variant per prediction set
VOTE_MEMBERS = None          # Part A span vote: list of run names, or None = every run with span predictions
PROB_ENSEMBLES = {}          # Part B probability-averaging ensembles you define yourself:
                             #   {"name": ["run_a", "run_b", ...]} -> every alignment x decoding x threshold variant,
                             #                                        scored on validation
                             #   {"name": {"members": [...], "align": "token", "decode": "viterbi", "min_frac": 0.2}}
                             #       -> only that variant (for ensembles of train_on_all runs, which cannot be
                             #          validated: use the settings that won for their validated twins)
GREEDY = True                # Part B: also build ensembles automatically by greedy forward selection on validation
REINFER = "all"              # Part B: earlier runs whose test/validation token scores are recomputed from best.pt:
                             #   "all" or a list of run names
RUN_DIR_OVERRIDES = {}       # {"swa_bitfit": "/kaggle/input/<...>/ner_runs/swa_bitfit"} — only if a run is found
                             #   in two inputs and the wrong copy is picked (N7 prints which folder each run uses)
TOP_N_ZIPS = 30              # zips written: this many best-on-validation sets + the best set of each member list

json.dump(CONFIG, open("/kaggle/working/run_config_ner.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
    f = e.get("epochs_from")
    assert not f or (e.get("train_on_all") and f in CONFIG["experiments"]), f"{name}: epochs_from needs train_on_all and a known run"
for name in SESSION_EXPERIMENTS:
    assert name in CONFIG["experiments"], f"SESSION_EXPERIMENTS: {name} is not defined"
n = len(SESSION_EXPERIMENTS)
print(f"{len(CONFIG['experiments'])} experiments defined; this session trains {n}: {SESSION_EXPERIMENTS}")
if n:
    print(f"rough GPU time: ~{n} x 10-20 min, two at a time on T4 x2 -> about {n * 15 / 2 / 60:.1f} h")

## N4. Pipeline code (notebook E + v2 additions at the end: Viterbi, propagation, ensembles, run finder)

In [ ]:
%%writefile ner_common.py
"""Phase 2 (L-NER) pipeline: data, label-balanced validation split, tokenisation with EXACT character offsets
(punctuation split before BPE), overlapping windows, BIO tags, span decoding, strict metrics (same rules as
evaluate_phase2.py), Phase-1 backbones that return hidden states, prediction and submission files.
v2 adds (end of file): BIO-constrained Viterbi decoding, document-level label propagation, probability-averaging
and span-voting ensembles, and a finder for the outputs of earlier runs."""
import glob, json, math, os, re, sys, zipfile
from collections import Counter
import numpy as np, torch, torch.nn as nn

INPUT_ROOT = os.environ.get("P2_INPUT", "/kaggle/input")      # read-only inputs (data prep, Phase-1 checkpoints)
WORK = os.environ.get("P2_WORK", "/kaggle/working")           # everything this notebook writes
DATA = f"{WORK}/p2data"                                        # Phase-2 files downloaded by cell N3

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

# ------------------------------------------------------------------------------------------------ labels / tags
LABELS = [x.strip() for x in open(f"{DATA}/labels.txt", encoding="utf-8") if x.strip()]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]     # BIO: 1 + 2 x 12 = 25 tags
TAG2ID = {t: i for i, t in enumerate(TAGS)}
IGNORE = -100                                                         # label for tokens that cover no character

# ------------------------------------------------------------------------------------------------ data
def load_json(name):
    return json.load(open(f"{DATA}/{name}", encoding="utf-8"))

def load_data():
    """texts[id] = str, gold[id] = [[start, end, label, text], ...] for the 70 training docs; test[id] = str."""
    texts, gold, fold = {}, {}, {}
    for f in ("fold1", "fold2"):
        t, g = load_json(f"{f}-text.json"), load_json(f"{f}-gold.json")
        texts.update({str(k): v for k, v in t.items()})
        gold.update({str(k): [list(e) for e in v] for k, v in g.items()})
        fold.update({str(k): f for k in t})
    test = {str(k): v for k, v in load_json("fold3-text.json").items()}
    return texts, gold, fold, test

def make_split(gold, n_val=14, target=0.2):
    """Greedy, deterministic: pick n_val documents so that every label has close to `target` of its entities
    in validation (documents are tried in sorted-id order, so the same split comes out every time)."""
    per_doc = {d: Counter(e[2] for e in ents) for d, ents in gold.items()}
    total = sum(per_doc.values(), Counter())
    chosen, cur = [], Counter()
    while len(chosen) < n_val:
        best = None
        for d in sorted(per_doc):
            if d in chosen:
                continue
            c = cur + per_doc[d]
            score = sum((c[l] / total[l] - target) ** 2 for l in LABELS if total[l])
            if best is None or score < best[0]:
                best = (score, d)
        chosen.append(best[1]); cur += per_doc[best[1]]
    val = sorted(chosen)
    train = sorted(d for d in gold if d not in chosen)
    share = {l: round(cur[l] / total[l], 3) if total[l] else None for l in LABELS}
    return train, val, share

# ------------------------------------------------------------------------------------------------ tokenisation
MARK_BYTES = len("▁".encode("utf-8"))
PIECE_RE = re.compile(r"\w+|[^\w\s]")

class SplitTokenizer:
    """Your Phase-1 BPE, applied to smaller pieces: each whitespace word is first split where letters/digits
    meet punctuation ("IPC," -> "IPC" + ","). The first piece keeps the word-start marker, later pieces don't,
    so tokens stay close to pretraining. Every token gets its exact (start, end) character span."""
    def __init__(self, bpe):
        self.bpe, self.stoi, self.unk = bpe, bpe.stoi, bpe.stoi["<unk>"]
        self.cache = {}

    def _pieces(self, s, first):
        key = (s, first)
        out = self.cache.get(key)
        if out is None:
            symbols = list(bpe_tok.word_to_symbols(("▁" if first else "") + s))
            while len(symbols) > 1:
                ranked = [(self.bpe.merge_rank[p], i) for i, p in enumerate(zip(symbols, symbols[1:]))
                          if p in self.bpe.merge_rank]
                if not ranked:
                    break
                _, i = min(ranked)
                symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]
            out = [(self.stoi.get(t, self.unk), len(t)) for t in symbols]
            if len(self.cache) < 2_000_000:
                self.cache[key] = out
        return out

    def encode(self, text):
        ids, spans, _, _ = self.encode_full(text)
        return ids, spans

    def encode_full(self, text):
        """ids, token spans, piece index of each token, and the (start, end) span of each piece
        (a piece = a run of letters/digits or one punctuation character inside a whitespace word)."""
        ids, spans, piece_of, piece_spans = [], [], [], []
        for m in re.finditer(r"\S+", text):
            w, ws = m.group(), m.start()
            for j, p in enumerate(PIECE_RE.finditer(w)):
                part, ps, first = p.group(), ws + p.start(), j == 0
                piece_spans.append((ps, ps + len(part)))
                raw, b = part.encode("utf-8"), 0
                for tid, nbytes in self._pieces(part, first):
                    k0 = max(0, b - (MARK_BYTES if first else 0)); b += nbytes
                    k1 = max(0, b - (MARK_BYTES if first else 0))
                    ids.append(tid)
                    spans.append((ps + len(raw[:k0].decode("utf-8", "ignore")),
                                   ps + len(raw[:k1].decode("utf-8", "ignore"))))
                    piece_of.append(len(piece_spans) - 1)
        return ids, spans, piece_of, piece_spans

def lead_tokens(spans, piece_of, n_pieces):
    """index of the first token of each piece that covers at least one character (-1 if none)"""
    lead = [-1] * n_pieces
    for i, ((s, e), p) in enumerate(zip(spans, piece_of)):
        if s < e and lead[p] < 0:
            lead[p] = i
    return lead

def piece_level_tags(tags, lead):
    """align="piece": only the first token of each piece keeps its tag (the rest are ignored in training)"""
    out = [IGNORE] * len(tags)
    for i in lead:
        if i >= 0:
            out[i] = tags[i]
    return out

def doc_tags(spans, ents):
    """BIO tag id per token. Tokens covering no character (a lone word-start marker) get IGNORE."""
    tags = [TAG2ID["O"] if s < e else IGNORE for s, e in spans]
    starts = np.array([s for s, _ in spans]); ends = np.array([e for _, e in spans])
    for s, e, l, *_ in ents:
        idx = np.nonzero((starts < e) & (ends > s) & (starts < ends))[0]
        for k, i in enumerate(idx):
            tags[i] = TAG2ID[("B-" if k == 0 else "I-") + l]
    return tags

def reachable(spans, ents):
    """How many gold entities can be reproduced exactly from token boundaries (the ceiling for strict F1)."""
    st, en = {s for s, e in spans if s < e}, {e for s, e in spans if s < e}
    return sum(1 for s, e, *_ in ents if s in st and e in en)

# ------------------------------------------------------------------------------------------------ windows
def windows(n, win, stride):
    """(start, end) token ranges covering 0..n; the last window is aligned to the end of the document."""
    if n <= win:
        return [(0, n)]
    out = list(range(0, n - win, stride)) + [n - win]
    return [(s, s + win) for s in sorted(set(out))]

def train_windows(docs, win, stride):
    """docs: {id: (ids, tags)} -> list of (ids_list, tags_list) windows."""
    out = []
    for d, (ids, tags) in docs.items():
        if not ids:
            continue
        for s, e in windows(len(ids), win, stride):
            out.append((ids[s:e], tags[s:e]))
    return out

def batches(wins, batch_size, shuffle, rng):
    """Batches of windows with EQUAL length (no padding: your models have no padding mask)."""
    by_len = {}
    for w in wins:
        by_len.setdefault(len(w[0]), []).append(w)
    out = []
    for L_, group in by_len.items():
        idx = rng.permutation(len(group)) if shuffle else np.arange(len(group))
        for k in range(0, len(group), batch_size):
            out.append([group[i] for i in idx[k:k + batch_size]])
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out

@torch.no_grad()
def predict_logits(model, ids, win, stride, device, batch_size=8):
    """Tag scores for every token of one document. Each token's scores come from the window in which it has
    the most context on both sides (its distance to the nearest window edge is largest)."""
    n = len(ids)
    out = np.zeros((n, len(TAGS)), dtype=np.float32)
    if n == 0:                                         # empty document: nothing to predict
        return out
    W = windows(n, win, stride)
    best = np.full(n, -1)
    x_all = torch.tensor(ids, dtype=torch.long)
    for k in range(0, len(W), batch_size):
        chunk = W[k:k + batch_size]
        x = torch.stack([x_all[s:e] for s, e in chunk]).to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x).float().cpu().numpy()
        for j, (s, e) in enumerate(chunk):
            pos = np.arange(s, e)
            left = pos - s if s > 0 else np.full(e - s, n)
            right = e - 1 - pos if e < n else np.full(e - s, n)
            score = np.minimum(left, right)
            take = score > best[s:e]
            out[s:e][take] = logits[j][take]
            best[s:e][take] = score[take]
    return out

# ------------------------------------------------------------------------------------------------ decoding
def decode(tag_ids, spans, mode="lenient"):
    """Token tags -> [[start, end, label], ...] character spans.
    lenient: an I-X that does not continue an X entity starts a new X entity.
    strict:  such an orphan I-X is ignored (only B-X can start an entity)."""
    ents, cur = [], None
    for t, (s, e) in zip(tag_ids, spans):
        if s >= e:
            continue                                   # token covers no character
        name = TAGS[t]
        if name == "O":
            if cur: ents.append(cur); cur = None
            continue
        bio, lab = name.split("-", 1)
        if bio == "I" and cur is not None and cur[2] == lab:
            cur[1] = e
            continue
        if cur: ents.append(cur); cur = None
        if bio == "B" or mode == "lenient":
            cur = [s, e, lab]
    if cur:
        ents.append(cur)
    seen, out = set(), []
    for s, e, l in ents:                               # the evaluator rejects duplicate spans
        if (s, e, l) not in seen:
            seen.add((s, e, l)); out.append([int(s), int(e), l])
    return out

# ------------------------------------------------------------------------------------------------ metrics
def strict_scores(gold, pred):
    """Same rules as evaluate_phase2.py: exact (start, end, label) match, macro over the 12 labels."""
    stats = {l: [0, 0, 0] for l in LABELS}
    for d in gold:
        g = {(s, e, l) for s, e, l, *_ in gold[d]}
        p = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        for l in LABELS:
            gs, ps = {x for x in g if x[2] == l}, {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = {"P": pr, "R": rc, "F1": 2 * pr * rc / (pr + rc) if pr + rc else 0.0, "tp": tp, "fp": fp, "fn": fn}
    mean = lambda k: float(np.mean([per[l][k] for l in LABELS]))
    return {"macro_f1": mean("F1"), "macro_p": mean("P"), "macro_r": mean("R"), "per_label": per}

def error_breakdown(gold, pred):
    """Classify every mistake: right span + wrong label, right label + wrong boundary, spurious, missed."""
    c = Counter()
    for d in gold:
        g = {(s, e, l) for s, e, l, *_ in gold[d]}; p = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        g_sp = {(s, e): l for s, e, l in g}
        for s, e, l in p - g:
            if (s, e) in g_sp:
                c["wrong label (span exact)"] += 1
            elif any(l2 == l and s < e2 and s2 < e for s2, e2, l2 in g):
                c["wrong boundary (label right)"] += 1
            else:
                c["spurious entity"] += 1
        p_sp = {(s, e) for s, e, _ in p}
        for s, e, l in g - p:
            if (s, e) not in p_sp and not any(l2 == l and s < e2 and s2 < e for s2, e2, l2 in p):
                c["missed entity"] += 1
    return dict(c)

# ------------------------------------------------------------------------------------------------ backbones
TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=1, n_heads=4, max_len=256, max_tgt_len=32, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

class Backbone(nn.Module):
    """Wraps one of YOUR Phase-1 models so that backbone(ids) returns hidden states (B, T, d_model) instead of
    next-token scores. Your model code is not changed: its output layer is replaced by Identity, so forward()
    returns the final-LayerNorm output. For encdec the ENCODER output is used (read with a hook).
    context="echo" (causal models): the window is fed twice and the second copy's states are used, so every
    token has seen the whole window (Springer et al. 2024, "Repetition Improves Language Model Embeddings")."""
    def __init__(self, arch, lm, bos_id, context="plain"):
        super().__init__()
        self.arch, self.lm, self.bos_id, self.context = arch, lm, bos_id, context
        self.kind = getattr(lm, "kind", "causal")
        self.d_model = lm.tok_emb.embedding_dim
        self.max_input = lm.max_len // 2 if context == "echo" else lm.max_len
        lm.head = nn.Identity()
        if arch == "encdec":
            # the NER head reads the ENCODER only: drop the decoder parts (unused, would waste memory and be
            # counted as trainable parameters)
            del lm.dec_layers, lm.dec_norm, lm.tgt_pos_emb

    def forward(self, ids):
        T0 = ids.size(1)
        x = torch.cat([ids, ids], 1) if self.context == "echo" else ids
        T = x.size(1)
        if self.arch == "encdec":                 # your EncDecLM's encoder path: embeddings -> enc_layers -> enc_norm
            lm = self.lm
            h = lm.tok_emb(x) + lm.src_pos_emb(torch.arange(T, device=x.device))
            for layer in lm.enc_layers:
                h = layer(h)
            h = lm.enc_norm(h)
        elif self.kind == "prefix":
            h = self.lm(x, prefix_len=T)          # whole input = prefix -> fully bidirectional
        else:
            h = self.lm(x)
        return h[:, T - T0:] if self.context == "echo" else h

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=len(bpe_tok_bpe().itos), **cfg)

_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

def load_backbone(arch, ckpt_pattern, context="plain", smoke=False):
    """Rebuilds your Phase-1 model from its checkpoint (its saved cfg + weights) and wraps it."""
    bpe = bpe_tok_bpe()
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return Backbone(arch, lm, bpe.stoi["<s>"], context)

def iter_adaptable(module, skip=("head",)):
    """Every weight a low-rank method (LoRA / LoRMA) can adapt inside `module`:
    yields (name, owner_module, parameter_name, weight) for nn.Linear weights and the packed q/k/v weight of
    nn.MultiheadAttention ('in_proj_weight'). Names containing an entry of `skip` are left out."""
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    """(attribute name, list of transformer layers, final LayerNorm) of the part of the model the NER head reads:
    decoder/prefix/swa/diffusion -> lm.layers + lm.norm ; moe -> lm.blocks + lm.norm ; encdec -> lm.enc_layers + lm.enc_norm"""
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

# ------------------------------------------------------------------------------------------------ submission
def write_predictions(preds, ids, path_json, path_zip=None):
    out = {d: preds.get(d, []) for d in ids}
    json.dump(out, open(path_json, "w", encoding="utf-8"), ensure_ascii=False)
    if path_zip:
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

# ================================================================================================ v2 additions
# ------------------------------------------------------------------------------------------------ Viterbi decoding
def _bio_transitions():
    """allowed[i, j] = tag j may follow tag i under BIO: I-X only after B-X or I-X. A sequence may not start with I-X.
    Returned as additive scores: 0 = allowed, -1e9 = forbidden."""
    T = len(TAGS)
    allowed = np.ones((T, T), dtype=bool)
    start = np.ones(T, dtype=bool)
    for j, t in enumerate(TAGS):
        if t.startswith("I-"):
            allowed[:, j] = False
            allowed[TAG2ID["B-" + t[2:]], j] = True
            allowed[j, j] = True
            start[j] = False
    return np.where(allowed, 0.0, -1e9).astype(np.float32), np.where(start, 0.0, -1e9).astype(np.float32)

TRANS, START = _bio_transitions()

def log_softmax(x):
    x = np.asarray(x, dtype=np.float32)
    m = x.max(-1, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(-1, keepdims=True))

def viterbi(logp):
    """The tag sequence with the highest total log-probability among sequences that are valid BIO
    (no I-X after O or after another label). No learned parameters: the model's per-token scores are kept,
    only invalid transitions are forbidden. logp: (n, n_tags) log-probabilities."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    back = np.zeros((n, T), dtype=np.int32)
    score = logp[0] + START
    cols = np.arange(T)
    for t in range(1, n):
        cand = score[:, None] + TRANS                 # cand[i, j]: best path ending in i, then tag j
        back[t] = cand.argmax(0)
        score = cand[back[t], cols] + logp[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

def positions(enc_doc, lead_doc, align):
    """Which tokens are decoded and the character span each one stands for.
    token: every token that covers a character; piece: the first token of each piece, standing for the whole piece."""
    ids, spans, piece_of, piece_spans = enc_doc
    if align == "piece":
        keep = [(i, sp) for i, sp in zip(lead_doc, piece_spans) if i >= 0]
    else:
        keep = [(i, sp) for i, sp in enumerate(spans) if sp[0] < sp[1]]
    return np.array([i for i, _ in keep], dtype=np.int64), [sp for _, sp in keep]

def decode_scores(scores, enc_doc, lead_doc, align="token", mode="lenient"):
    """Token scores (n_tokens, n_tags), logits or log-probabilities -> [[start, end, label], ...].
    mode: "lenient" / "strict" = argmax per token, then the BIO repair of decode();
          "viterbi" = best BIO-valid sequence over the whole document."""
    idx, sps = positions(enc_doc, lead_doc, align)
    if len(idx) == 0:
        return []
    sc = np.asarray(scores, dtype=np.float32)[idx]
    if mode == "viterbi":
        return decode(viterbi(log_softmax(sc)), sps, "strict")
    return decode(sc.argmax(-1), sps, mode)

# ------------------------------------------------------------------------------------------------ propagation
def propagate(text, ents, min_frac=0.2, min_len=2):
    """Document-level label consistency (Krishnan & Manning 2006). In the training gold, every case-sensitive
    occurrence of an annotated string carries the same label, unless it lies inside a longer entity.
    For each string the model predicted at least once:
      share = (times it was predicted) / (times the string occurs in the document)
    if share >= min_frac, every occurrence becomes an entity with the string's majority label.
    Overlaps are resolved longest-first; strings below the threshold keep their original predictions."""
    votes = {}
    for s, e, l, *_ in ents:
        votes.setdefault(text[s:e], Counter())[l] += 1
    cands = {(s, e): l for s, e, l, *_ in ents}
    for string, c in votes.items():
        if len(string.strip()) < min_len:
            continue
        occ = [m.start() for m in re.finditer(re.escape(string), text)]
        if not occ or sum(c.values()) / len(occ) < min_frac:
            continue
        lab = c.most_common(1)[0][0]
        for s in occ:
            cands[(s, s + len(string))] = lab
    taken = np.zeros(len(text) + 1, dtype=bool)
    out = []
    for (s, e), l in sorted(cands.items(), key=lambda kv: (kv[0][0] - kv[0][1], kv[0][0])):
        if not taken[s:e].any():
            taken[s:e] = True
            out.append([int(s), int(e), l])
    return sorted(out)

# ------------------------------------------------------------------------------------------------ ensembles
def mean_logprobs(score_list):
    """Probability-averaging ensemble: mean of the members' softmax distributions, returned as log-probabilities."""
    p = sum(np.exp(log_softmax(s)) for s in score_list) / len(score_list)
    return np.log(p + 1e-9).astype(np.float32)

def span_vote(pred_lists, k):
    """Span-level vote over members' predictions: keep a (start, end) predicted by at least k members, with its
    most frequent label. Overlapping survivors: more votes first, then longer."""
    votes, labs = Counter(), {}
    for p in pred_lists:
        for s, e, l in {(x[0], x[1], x[2]) for x in p}:
            votes[(s, e)] += 1
            labs.setdefault((s, e), Counter())[l] += 1
    out, used = [], []
    for (s, e), n in sorted(votes.items(), key=lambda kv: (-kv[1], kv[0][0] - kv[0][1], kv[0][0])):
        if n < k or any(s < e2 and s2 < e for s2, e2 in used):
            continue
        used.append((s, e))
        out.append([int(s), int(e), labs[(s, e)].most_common(1)[0][0]])
    return sorted(out)

# ------------------------------------------------------------------------------------------------ earlier runs
def find_runs(filename, overrides=None):
    """{experiment: folder} for every '.../ner_runs/<experiment>/<filename>' (or '.../logits/<experiment>/...')
    under /kaggle/working and /kaggle/input. If an experiment is found more than once, /kaggle/working wins,
    then the newest file. overrides = {experiment: folder} pins a folder by hand."""
    hits = {}
    for rank, root in enumerate((WORK, INPUT_ROOT)):
        for p in glob.glob(f"{root}/**/{filename}", recursive=True):
            folder = os.path.dirname(p)
            if os.path.basename(os.path.dirname(folder)) not in ("ner_runs", "logits"):
                continue
            exp, key = os.path.basename(folder), (rank, -os.path.getmtime(p))
            if exp not in hits or key < hits[exp][0]:
                hits[exp] = (key, folder)
    out = {e: f for e, (_, f) in sorted(hits.items())}
    out.update(overrides or {})
    return out

def run_info(folder):
    """align / train_on_all of a finished run (info.json written by v2, else the run's summary.json)."""
    for name in ("info.json", "summary.json"):
        if os.path.exists(f"{folder}/{name}"):
            s = json.load(open(f"{folder}/{name}"))
            return {"align": s.get("align", "token"), "train_on_all": bool(s.get("train_on_all", False))}
    return {"align": "token", "train_on_all": False}

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `ner_methods.py` — YOUR CODE (unchanged copy from notebook E)

In [ ]:
%%writefile ner_methods.py
# =====================================================================================================
#  CS-781 Phase 2 — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import ner_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}

## N7. Scripts: training (one experiment), re-inference (one earlier run), the lab, the official evaluator
`train_ner.py` is notebook E's script plus: per-experiment `seed`, `epochs_from`, `decode`/`propagate` options, and it
saves validation + test token scores. `infer_ner.py` rebuilds an earlier run from its `best.pt` (the settings are stored
inside it) and saves the same token scores.

In [ ]:
%%writefile train_ner.py
"""Runs ONE experiment from run_config_ner.json: builds your Phase-1 backbone + your NER head, applies your
fine-tuning method, trains on the training documents, keeps the epoch with the best validation strict
macro-F1, then writes validation + test predictions and the Codabench zip.
v2: per-experiment "seed"; "epochs_from" (train_on_all runs reuse the best epoch of a validation run);
"decode" can be "viterbi"; "propagate" applies document-level label propagation; validation AND test token
scores are saved (val_logits.npz, test_logits.npz) so the lab (N11) can re-decode and ensemble on CPU.
   python train_ner.py --exp NAME --device cuda:0        (real run)
   python train_ner.py --exp NAME --device cpu --smoke   (tiny random model, few documents: checks the code)"""
import argparse, json, math, os, random, time
import numpy as np, torch, torch.nn.functional as F
import ner_common as L
import ner_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
if dev.startswith("cuda"):
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_ner.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        return {**C["defaults"], **C["experiments"][name], "name": name}
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
RUNS = f"{L.WORK}/{'ner_runs_smoke' if args.smoke else 'ner_runs'}"

def best_epoch_of(name):
    """best validation epoch of a finished run (this session, or an earlier one added as input)"""
    p = f"{RUNS}/{name}/summary.json"
    if not os.path.exists(p):
        p = f"{L.find_runs('summary.json')[name]}/summary.json"
    return int(json.load(open(p))["best_epoch"])

if E["train_on_all"] and E.get("epochs_from") and not args.smoke:
    E["epochs"] = best_epoch_of(E["epochs_from"])
if args.smoke:
    E.update(window=96, stride=64, epochs=1, batch_size=4, patience=99)
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = E.get("seed", C["seed"]); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ data
texts, gold, fold, test = L.load_data()
train_ids, val_ids, share = L.make_split(gold, n_val=C["n_val_docs"], target=C["val_fraction"])
if E["train_on_all"]:
    train_ids = sorted(gold)
if args.smoke:
    train_ids, val_ids, test = train_ids[:4], val_ids[:2], dict(list(test.items())[:2])
all_text = {**texts, **test}
tok = L.SplitTokenizer(L.bpe_tok_bpe())
enc = {d: tok.encode_full(t) for d, t in all_text.items() if d in set(train_ids) | set(val_ids) | set(test)}
lead = {d: L.lead_tokens(enc[d][1], enc[d][2], len(enc[d][3])) for d in enc}
def train_tags(d):
    t = L.doc_tags(enc[d][1], gold[d])
    return L.piece_level_tags(t, lead[d]) if E["align"] == "piece" else t
train_docs = {d: (enc[d][0], train_tags(d)) for d in train_ids}

# ------------------------------------------------------------------ model
def build(e, smoke):
    backbone = L.load_backbone(e["arch"], C["checkpoints"][e["arch"]], context=e["context"], smoke=smoke)
    if e.get("student_layers"):
        backbone = M.make_student(backbone, e["student_layers"])
    model = M.TokenClassifier(backbone, n_tags=len(L.TAGS), dropout=e["head_dropout"])
    for p in model.parameters():
        p.requires_grad_(False)
    M.METHODS[e["method"]](model, **e["method_cfg"])
    for p in model.head.parameters():          # the new head is always trained
        p.requires_grad_(True)
    return model

model = build(E, args.smoke).to(dev)
def fit_window(max_input):
    """window capped at what the model accepts; the stride keeps the same overlap ratio as configured"""
    w = min(E["window"], max_input)
    return w, max(1, min(E["stride"], round(w * E["stride"] / E["window"])))
win, stride = fit_window(model.backbone.max_input)
total, trainable = L.count_params(model)
say(event="start", arch=E["arch"], method=E["method"], params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), window=win, stride=stride, epochs=E["epochs"], seed=seed,
    decode=E["decode"], propagate=E.get("propagate"),
    train_docs=len(train_ids), val_docs=len(val_ids), val_label_share=share)

teacher = None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = L.find_input(f"ner_runs/{E['teacher']}/best.pt")      # teacher trained in an earlier session
    teacher = build(TE, args.smoke)
    teacher.load_state_dict(torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    win, stride = fit_window(min(win, teacher.backbone.max_input))
    say(event="teacher", name=E["teacher"], params_total=L.count_params(teacher)[0])

# ------------------------------------------------------------------ training setup
wins = L.train_windows(train_docs, win, stride)
tag_counts = np.bincount([t for _, tags in wins for t in tags if t != L.IGNORE], minlength=len(L.TAGS))
weights = None
if E["class_weights"] == "inv_sqrt":
    w = 1.0 / np.sqrt(np.maximum(tag_counts, 1)); w = w / w.mean()
    weights = torch.tensor(w, dtype=torch.float32, device=dev)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
rng = np.random.RandomState(seed)
steps_per_epoch = len(L.batches(wins, E["batch_size"], False, rng))
if args.smoke:
    steps_per_epoch = min(steps_per_epoch, 6)
total_steps = steps_per_epoch * E["epochs"]
warm = max(1, int(E["warmup_frac"] * total_steps))
def lr_at(step):                                   # linear warm-up, then linear decay to 0
    return E["lr"] * min((step + 1) / warm, max(0.0, (total_steps - step) / max(1, total_steps - warm)))
scaler = torch.amp.GradScaler("cuda", enabled=dev.startswith("cuda"))

def decode_doc(d, logits):
    ents = L.decode_scores(logits, enc[d], lead[d], E["align"], E["decode"])
    if E.get("propagate") is not None:
        ents = L.propagate(all_text[d], ents, E["propagate"])
    return ents

def evaluate(doc_ids, keep_logits=None):
    model.eval()
    preds = {}
    for d in doc_ids:
        logits = L.predict_logits(model, enc[d][0], win, stride, dev, E["batch_size"])
        preds[d] = decode_doc(d, logits)
        if keep_logits is not None:
            keep_logits[d] = logits.astype(np.float16)
    model.train()
    return preds

# ------------------------------------------------------------------ train
best, best_epoch, bad, step = -1.0, -1, 0, 0
model.train()
if dev.startswith("cuda"):
    torch.cuda.reset_peak_memory_stats()
for epoch in range(1, E["epochs"] + 1):
    te, losses = time.time(), []
    for bi, batch in enumerate(L.batches(wins, E["batch_size"], True, rng)):
        if args.smoke and bi >= steps_per_epoch:
            break
        x = torch.tensor([w[0] for w in batch], dtype=torch.long, device=dev)
        y = torch.tensor([w[1] for w in batch], dtype=torch.long, device=dev)
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x).float()
            if teacher is not None:
                with torch.no_grad():
                    t_logits = teacher(x).float()
                loss = M.kd_loss(logits, t_logits, y, **E["kd"])
            else:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1),
                                       weight=weights, ignore_index=L.IGNORE)
            aux = getattr(model.backbone.lm, "aux_loss", None)
            if E["moe_aux_coef"] and aux is not None:
                loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); step += 1
    epoch_sec = time.time() - te
    if E["train_on_all"]:
        say(event="epoch", epoch=epoch, train_loss=round(float(np.mean(losses)), 4), sec=round(epoch_sec))
        torch.save({"state": model.state_dict(), "exp": E, "epoch": epoch}, f"{OUT}/best.pt"); best_epoch = epoch
        continue
    s = L.strict_scores({d: gold[d] for d in val_ids}, evaluate(val_ids))
    say(event="epoch", epoch=epoch, train_loss=round(float(np.mean(losses)), 4), val_macro_f1=round(s["macro_f1"], 4),
        val_macro_p=round(s["macro_p"], 4), val_macro_r=round(s["macro_r"], 4), sec=round(epoch_sec))
    if s["macro_f1"] > best:
        best, best_epoch, bad = s["macro_f1"], epoch, 0
        torch.save({"state": model.state_dict(), "exp": E, "epoch": epoch}, f"{OUT}/best.pt")
    else:
        bad += 1
        if bad >= E["patience"]:
            say(event="early_stop", epoch=epoch)
            break

# ------------------------------------------------------------------ final predictions with the best epoch
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev, weights_only=False)["state"])
peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if dev.startswith("cuda") else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "align": E["align"], "decode": E["decode"],
           "propagate": E.get("propagate"), "seed": seed, "train_on_all": E["train_on_all"], "epochs": E["epochs"],
           "best_epoch": best_epoch, "params_total": total, "params_trainable": trainable, "peak_gpu_gb": peak,
           "train_windows": len(wins), "window": win, "stride": stride}
if not E["train_on_all"]:
    vlog = {}
    vpred = evaluate(val_ids, vlog)
    np.savez_compressed(f"{OUT}/val_logits.npz", **vlog)      # for the decoding / ensembling lab (N11)
    L.write_predictions(vpred, val_ids, f"{OUT}/val_prediction.json")
    json.dump({d: [[s, e, l] for s, e, l, *_ in gold[d]] for d in val_ids}, open(f"{OUT}/val_truth.json", "w"))
    s = L.strict_scores({d: gold[d] for d in val_ids}, vpred)
    summary.update(val_macro_f1=round(s["macro_f1"], 4), val_macro_p=round(s["macro_p"], 4), val_macro_r=round(s["macro_r"], 4),
                   val_per_label_f1={l: round(v["F1"], 4) for l, v in s["per_label"].items()},
                   val_errors=L.error_breakdown({d: gold[d] for d in val_ids}, vpred))
ti = time.time()
tlog = {}
tpred = evaluate(list(test), tlog)
np.savez_compressed(f"{OUT}/test_logits.npz", **tlog)
summary["test_predict_sec"] = round(time.time() - ti, 1)
L.write_predictions(tpred, list(test), f"{OUT}/prediction.json", f"{OUT}/submission_{args.exp}.zip")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
json.dump({"exp": args.exp, "align": E["align"], "train_on_all": E["train_on_all"], "window": win, "stride": stride,
           "source": f"{OUT}/best.pt"}, open(f"{OUT}/info.json", "w"), indent=1)
say(event="done", **{k: v for k, v in summary.items() if k not in ("val_per_label_f1", "val_errors")})

In [ ]:
%%writefile infer_ner.py
"""Re-computes the token scores (validation + test documents) of a finished experiment from its saved
checkpoint (ner_runs/<experiment>/best.pt of an earlier session), so it can be re-decoded and ensembled in the
lab (N11) without retraining. The model is rebuilt from the settings stored inside the checkpoint, plus the
Phase-1 checkpoint of its architecture (cell N3 "checkpoints").
   python infer_ner.py --run FOLDER --out FOLDER --device cuda:0"""
import argparse, json, os, time
import numpy as np, torch
import ner_common as L
import ner_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True); ap.add_argument("--out", required=True)
ap.add_argument("--device", default="cuda:0")
args = ap.parse_args()
dev = args.device
if dev.startswith("cuda"):
    torch.cuda.set_device(dev)
t0 = time.time()

C = json.load(open(f"{L.WORK}/run_config_ner.json"))
ck = torch.load(f"{args.run}/best.pt", map_location="cpu", weights_only=False)
E = ck["exp"]
name = os.path.basename(os.path.normpath(args.run))
backbone = L.load_backbone(E["arch"], C["checkpoints"][E["arch"]], context=E.get("context", "plain"))
if E.get("student_layers"):
    backbone = M.make_student(backbone, E["student_layers"])
model = M.TokenClassifier(backbone, n_tags=len(L.TAGS), dropout=E.get("head_dropout", 0.1))
M.METHODS[E["method"]](model, **E.get("method_cfg", {}))
model.load_state_dict(ck["state"])
model.to(dev).eval()

# the same window / stride the run used (its summary.json), else the configured ones capped at the model's limit
summ = json.load(open(f"{args.run}/summary.json")) if os.path.exists(f"{args.run}/summary.json") else {}
win = summ.get("window") or min(E["window"], model.backbone.max_input)
stride = summ.get("stride") or max(1, min(E["stride"], round(win * E["stride"] / E["window"])))

texts, gold, fold, test = L.load_data()
_, val_ids, _ = L.make_split(gold, n_val=C["n_val_docs"], target=C["val_fraction"])
tok = L.SplitTokenizer(L.bpe_tok_bpe())
os.makedirs(args.out, exist_ok=True)
splits = [("test", list(test), test)] + ([] if E.get("train_on_all") else [("val", val_ids, texts)])
for split, ids, src in splits:
    out = {d: L.predict_logits(model, tok.encode(src[d])[0], win, stride, dev).astype(np.float16) for d in ids}
    np.savez_compressed(f"{args.out}/{split}_logits.npz", **out)
json.dump({"exp": name, "align": E.get("align", "token"), "train_on_all": bool(E.get("train_on_all")),
           "window": win, "stride": stride, "epoch": ck.get("epoch"), "source": f"{args.run}/best.pt"},
          open(f"{args.out}/info.json", "w"), indent=1)
print(f"{name}: scores for {', '.join(sp for sp, _, _ in splits)} written to {args.out} in {time.time() - t0:.0f}s")

In [ ]:
%%writefile ner_lab.py
"""Phase 2 v2 — decoding / post-processing / ensembling lab. No training: it takes what finished experiments
saved (span predictions, or token scores), builds new prediction sets from them, scores each one on the SAME
14 validation documents as every experiment, and writes the test predictions as Codabench zips.
Every variant is recorded in submissions/index.csv; you choose which zips to upload."""
import csv, glob, hashlib, json, os, time
import numpy as np
import ner_common as L


class Lab:
    def __init__(self, n_val, val_fraction, out_dir=f"{L.WORK}/submissions"):
        self.texts, self.gold, _, self.test = L.load_data()
        _, self.val_ids, _ = L.make_split(self.gold, n_val=n_val, target=val_fraction)
        self.G = {d: self.gold[d] for d in self.val_ids}
        self.test_ids = list(self.test)
        self.out_dir = out_dir
        os.makedirs(f"{out_dir}/val_predictions", exist_ok=True)
        json.dump({d: [[s, e, l] for s, e, l, *_ in self.G[d]] for d in self.val_ids},
                  open(f"{out_dir}/val_predictions/val_truth.json", "w"))
        self.rows, self.models, self.enc, self.lead = [], {}, None, None

    def text(self, d):
        return self.texts[d] if d in self.texts else self.test[d]

    def add(self, name, kind, val_preds, test_preds, write=True, **info):
        """record one prediction set: validation scores (None if it has no validation predictions) + test preds"""
        s = L.strict_scores(self.G, val_preds) if val_preds is not None else None
        row = {"name": name, "kind": kind,
               "val_f1": round(s["macro_f1"], 4) if s else None, "val_p": round(s["macro_p"], 4) if s else None,
               "val_r": round(s["macro_r"], 4) if s else None, **info}
        self.rows.append((row, val_preds, test_preds if write else None))
        return row["val_f1"]

    def prop(self, preds, min_frac):
        return None if preds is None else {d: L.propagate(self.text(d), p, min_frac) for d, p in preds.items()}

    def add_with_prop(self, name, kind, v, t, prop_grid, **info):
        """the prediction set itself + one variant per propagation threshold"""
        self.add(name, kind, v, t, min_frac=None, **info)
        for mf in prop_grid:
            self.add(f"{name}__prop{mf}", kind, self.prop(v, mf), self.prop(t, mf), min_frac=mf, **info)

    # ------------------------------------------------------------------ Part A: span predictions of earlier runs
    def span_level(self, prop_grid, vote_members=None, overrides=None):
        runs = {e: f for e, f in L.find_runs("val_prediction.json", overrides).items()
                if os.path.exists(f"{f}/prediction.json")}
        preds = {e: (json.load(open(f"{f}/val_prediction.json")), json.load(open(f"{f}/prediction.json")))
                 for e, f in runs.items()}
        print(f"Part A: span predictions found for {len(preds)} runs:", ", ".join(preds) or "none")
        for e, (v, t) in preds.items():
            # the run's own submission is recorded for comparison but not written again (already submitted)
            self.add(f"A_{e}", "A single (as submitted)", v, t, write=False, members=e, min_frac=None)
            for mf in prop_grid:
                self.add(f"A_{e}__prop{mf}", "A single + propagation", self.prop(v, mf), self.prop(t, mf),
                         members=e, min_frac=mf)
        members = [m for m in (vote_members or list(preds)) if m in preds]
        for k in range(2, len(members)):
            v = {d: L.span_vote([preds[m][0].get(d, []) for m in members], k) for d in self.val_ids}
            t = {d: L.span_vote([preds[m][1].get(d, []) for m in members], k) for d in self.test_ids}
            self.add_with_prop(f"A_vote{k}of{len(members)}", "A span vote", v, t, prop_grid, members="+".join(members))

    # ------------------------------------------------------------------ Part B: token scores
    def load_scores(self, overrides=None):
        runs = L.find_runs("test_logits.npz", overrides)
        for e, f in runs.items():
            info = L.run_info(f)
            has_val = os.path.exists(f"{f}/val_logits.npz") and not info["train_on_all"]
            z_t = np.load(f"{f}/test_logits.npz")
            z_v = np.load(f"{f}/val_logits.npz") if has_val else None
            self.models[e] = {"align": info["align"], "folder": f,
                              "test": {d: z_t[d].astype(np.float32) for d in self.test_ids},
                              "val": {d: z_v[d].astype(np.float32) for d in self.val_ids} if has_val else None}
        if self.models and self.enc is None:
            t0 = time.time()
            tok = L.SplitTokenizer(L.bpe_tok_bpe())
            self.enc = {d: tok.encode_full(self.text(d)) for d in self.val_ids + self.test_ids}
            self.lead = {d: L.lead_tokens(e[1], e[2], len(e[3])) for d, e in self.enc.items()}
            print(f"tokenised {len(self.enc)} documents in {time.time() - t0:.0f}s")
        for e, m in self.models.items():
            n_tok = len(self.enc[self.test_ids[0]][0]) if self.enc else 0
            assert m["test"][self.test_ids[0]].shape[0] == n_tok, f"{e}: token scores do not match the tokenizer"
        print(f"Part B: token scores found for {len(self.models)} runs:",
              ", ".join(f"{e}{'' if m['val'] is not None else ' (no validation)'}" for e, m in self.models.items()) or "none")

    def aligns_for(self, members):
        """a model trained with align='piece' has untrained scores on non-first tokens -> piece decoding only"""
        return ["piece"] if any(self.models[m]["align"] == "piece" for m in members) else ["token", "piece"]

    def decode(self, members, split, align, mode):
        ids = self.val_ids if split == "val" else self.test_ids
        if split == "val" and any(self.models[m]["val"] is None for m in members):
            return None
        out = {}
        for d in ids:
            sc = [self.models[m][split][d] for m in members]
            out[d] = L.decode_scores(sc[0] if len(sc) == 1 else L.mean_logprobs(sc), self.enc[d], self.lead[d], align, mode)
        return out

    def score_variants(self, name, kind, members, prop_grid, modes):
        for align in self.aligns_for(members):
            for mode in modes:
                v, t = self.decode(members, "val", align, mode), self.decode(members, "test", align, mode)
                self.add_with_prop(f"{name}__{align}-{mode}", kind, v, t, prop_grid,
                                   members="+".join(members), align=align, decode=mode)

    def singles(self, prop_grid, modes=("lenient", "viterbi")):
        for e, m in self.models.items():
            if m["val"] is None:
                print(f"{e}: no validation scores (train_on_all) -> its zip is the one train_ner.py wrote")
                continue
            self.score_variants(f"B_{e}", "B single", [e], prop_grid, modes)

    def ensembles(self, groups, prop_grid, modes=("lenient", "viterbi")):
        """groups = {name: [members]} -> every alignment x mode x propagation variant, scored on validation;
        groups = {name: {"members": [...], "align": ..., "decode": ..., "min_frac": ...}} -> that one variant only
        (for ensembles of train_on_all runs, which cannot be validated: copy the settings of their validated twins)."""
        for name, g in groups.items():
            members = g["members"] if isinstance(g, dict) else list(g)
            missing = [m for m in members if m not in self.models]
            if missing:
                print(f"ensemble {name}: skipped, no token scores for {missing}")
                continue
            if not isinstance(g, dict):
                self.score_variants(f"B_ens_{name}", "B prob. ensemble", members, prop_grid, modes)
                continue
            v, t = (self.decode(members, sp, g["align"], g["decode"]) for sp in ("val", "test"))
            mf = g.get("min_frac")
            if mf is not None:
                v, t = self.prop(v, mf), self.prop(t, mf)
            self.add(f"B_ens_{name}__{g['align']}-{g['decode']}" + (f"__prop{mf}" if mf is not None else ""),
                     "B prob. ensemble", v, t, members="+".join(members), align=g["align"], decode=g["decode"], min_frac=mf)

    def greedy(self, prop_grid, modes=("lenient", "viterbi")):
        """Greedy forward ensemble selection (Caruana et al. 2004) on validation: start empty, repeatedly add the
        model that raises validation macro-F1 the most (best propagation threshold of the grid, or none),
        stop when nothing helps. Done separately for each alignment x decoding mode."""
        pool = [e for e, m in self.models.items() if m["val"] is not None]
        for align in ("token", "piece"):
            cands = [e for e in pool if align == "piece" or self.models[e]["align"] != "piece"]
            for mode in modes:
                def objective(members):
                    base = self.decode(members, "val", align, mode)
                    best = (L.strict_scores(self.G, base)["macro_f1"], None)
                    for mf in prop_grid:
                        f = L.strict_scores(self.G, self.prop(base, mf))["macro_f1"]
                        if f > best[0]:
                            best = (f, mf)
                    return best
                chosen, best = [], (-1.0, None)
                while True:
                    tries = [(objective(chosen + [e]), e) for e in cands if e not in chosen]
                    if not tries:
                        break
                    (f, mf), e = max(tries, key=lambda x: x[0][0])
                    if f <= best[0]:
                        break
                    chosen.append(e); best = (f, mf)
                if len(chosen) < 2:
                    print(f"greedy {align}-{mode}: no ensemble beats the best single model ({chosen})")
                    continue
                v, t = self.decode(chosen, "val", align, mode), self.decode(chosen, "test", align, mode)
                if best[1] is not None:
                    v, t = self.prop(v, best[1]), self.prop(t, best[1])
                self.add(f"B_greedy__{align}-{mode}" + (f"__prop{best[1]}" if best[1] is not None else ""),
                         "B greedy ensemble", v, t, members="+".join(chosen), align=align, decode=mode, min_frac=best[1])
                print(f"greedy {align}-{mode}: {chosen} -> validation macro-F1 {best[0]:.4f} (propagation {best[1]})")

    # ------------------------------------------------------------------ output
    def finish(self, top_n=30):
        """Writes zips for: the top_n prediction sets by validation macro-F1, the best set of each distinct
        member list, and every set that cannot be validated (fixed-setting ensembles of train_on_all runs).
        Identical test predictions are written once. Everything (written or not) is listed in index.csv."""
        scored = sorted([r for r in self.rows if r[0]["val_f1"] is not None], key=lambda r: -r[0]["val_f1"])
        keep = {id(r) for r in scored[:top_n]}
        best_of = {}
        for r in scored:
            best_of.setdefault((r[0]["kind"].split(" ")[0], r[0].get("members")), r)
        keep |= {id(r) for r in best_of.values()} | {id(r) for r in self.rows if r[0]["val_f1"] is None}
        order = scored + [r for r in self.rows if r[0]["val_f1"] is None]     # best first: it names the zip
        for f in glob.glob(f"{self.out_dir}/*.zip") + glob.glob(f"{self.out_dir}/val_predictions/*.json"):
            if not f.endswith("val_truth.json"):                    # files of an earlier finish() (e.g. Part A)
                os.remove(f)
        seen, n_written = {}, 0
        for r in order:
            row, v, t = r
            row["file"] = ""
            if t is None or id(r) not in keep:
                continue
            out = {d: t.get(d, []) for d in self.test_ids}
            h = hashlib.md5(json.dumps(out, sort_keys=True).encode()).hexdigest()
            if h in seen:
                row["file"] = f"(same as {seen[h]})"
                continue
            seen[h] = row["file"] = f"{row['name']}.zip"
            L.write_predictions(out, self.test_ids, f"{self.out_dir}/_prediction.json", f"{self.out_dir}/{row['file']}")
            if v is not None:
                json.dump(v, open(f"{self.out_dir}/val_predictions/{row['name']}.json", "w"))
            n_written += 1
        if os.path.exists(f"{self.out_dir}/_prediction.json"):
            os.remove(f"{self.out_dir}/_prediction.json")
        cols = ["name", "kind", "val_f1", "val_p", "val_r", "members", "align", "decode", "min_frac", "file"]
        with open(f"{self.out_dir}/index.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for row, _, _ in order:
                w.writerow(row)
        print(f"{len(self.rows)} prediction sets scored, {n_written} zips written to {self.out_dir}/ (index.csv lists all)")
        return [r[0] for r in self.rows]


def table(rows, n=25):
    """the n best prediction sets on validation, then any written set that has no validation score"""
    scored = sorted([r for r in rows if r["val_f1"] is not None], key=lambda r: -r["val_f1"])
    print(f"{'val F1':>7s} {'P':>6s} {'R':>6s}  {'prediction set':70s} zip")
    for r in scored[:n]:
        print(f"{r['val_f1']:7.4f} {r['val_p']:6.3f} {r['val_r']:6.3f}  {r['name'][:70]:70s} {r.get('file', '')}")
    for r in rows:
        if r["val_f1"] is None and r.get("file"):
            print(f"{'n/a':>7s} {'':6s} {'':6s}  {r['name'][:70]:70s} {r['file']}")

In [ ]:
%%writefile evaluate_phase2.py
#!/usr/bin/env python3
"""Local evaluator for Phase 2.

Usage:
  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt
"""
import argparse, json, sys

class DuplicateJSONKeyError(ValueError): pass
def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")
def pairs_no_duplicates(pairs):
    out={}; dup=[]
    for k,v in pairs:
        if k in out: dup.append(k)
        out[k]=v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out
def load_json(path):
    with open(path,encoding='utf-8') as f:
        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e
def labels(path):
    with open(path,encoding='utf-8') as f: xs=[x.strip() for x in f if x.strip()]
    if not xs: raise ValueError('labels.txt is empty.')
    if len(xs)!=12: raise ValueError(f'Expected 12 labels, found {len(xs)}.')
    if len(set(xs))!=len(xs): raise ValueError('labels.txt contains duplicate labels.')
    return xs
def spans(x, allowed, where):
    if x is None: return set()
    if not isinstance(x,list): raise ValueError(f'{where}: expected a list of entities.')
    out=set()
    for j,s in enumerate(x):
        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].')
        a,b,l=s[:3]
        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f'{where}, entity {j}: start/end must be integers.')
        if a<0 or b<=a: raise ValueError(f'{where}, entity {j}: require 0 <= start < end.')
        if not isinstance(l,str) or l not in allowed: raise ValueError(f'{where}, entity {j}: invalid label {l!r}.')
        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f'{where}, entity {j}: entity_text must be a string.')
        t=(a,b,l)
        if t in out: raise ValueError(f'{where}, entity {j}: duplicate span {t}.')
        out.add(t)
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 2 entity predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); ap.add_argument('--labels',required=True)
    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)
    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError('Truth and prediction must be non-empty JSON objects.')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    stats={l:[0,0,0] for l in labs}; allowed=set(labs)
    for did in truth:
        g=spans(truth[did],allowed,f'truth[{did}]'); p=spans(pred[did],allowed,f'prediction[{did}]')
        for l in labs:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    precisions=[]; recalls=[]; f1s=[]
    print(f'Examples: {len(truth)}')
    for l in labs:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0
        precisions.append(pr); recalls.append(rc); f1s.append(f)
        print(f'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}')
    print(f'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}')
    print(f'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}')
    print(f'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

## N8. Part A — post-process the span predictions you already have (CPU, no checkpoints needed)
For every notebook-E run: its validation/test predictions + propagation at each threshold in `PROP_MIN_FRACS`; and a
span vote across runs (a span survives if at least k runs predicted it, k = 2 … n−1). Zips go to
`/kaggle/working/submissions/` (the as-submitted originals are scored but not written again).

In [ ]:
# Part A (CPU, ~1-2 min): span predictions your notebook-E runs already wrote (val_prediction.json + prediction.json)
#  -> document-level propagation at each threshold of PROP_MIN_FRACS, and span-level voting across runs.
import json, importlib, ner_common as L, ner_lab
importlib.reload(L); importlib.reload(ner_lab)
C = json.load(open("/kaggle/working/run_config_ner.json"))
for e, f in L.find_runs("val_prediction.json", RUN_DIR_OVERRIDES).items():
    print(f"  {e:24s} <- {f}")
lab = ner_lab.Lab(C["n_val_docs"], C["val_fraction"])
lab.span_level(PROP_MIN_FRACS, VOTE_MEMBERS, RUN_DIR_OVERRIDES)
rows = lab.finish(TOP_N_ZIPS)
ner_lab.table(rows, 15)

## N9. Part B — recompute token scores of the earlier runs from their checkpoints (GPU, ~1 min each)
Notebook E saved validation scores but not test scores, and Viterbi / probability ensembles need both. Runs that
already have them (this session's, or an earlier session of this notebook added as input) are skipped.

In [ ]:
# Part B (GPU, ~1 min per run): rebuild each earlier run from its best.pt and save its token scores for the
# validation and test documents -> /kaggle/working/logits/<run>/. Needs the Phase-1 checkpoints as inputs (N3).
import subprocess, sys, os, time, torch, ner_common as L
runs = L.find_runs("best.pt", RUN_DIR_OVERRIDES)
have = L.find_runs("test_logits.npz")
todo = [e for e in runs if (REINFER == "all" or e in REINFER) and e not in have]
print("runs with a checkpoint:", list(runs))
print("token scores already available:", list(have))
print("recomputing now:", todo)
os.makedirs("/kaggle/working/logits", exist_ok=True)
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
running, failed = {}, []
while todo or running:
    for dev in devices:
        if dev in running and running[dev][1].poll() is not None:
            e, p = running.pop(dev)
            out = open(f"/kaggle/working/logits/{e}.out").read()
            lines = [l for l in out.splitlines() if "warn" not in l.lower()]
            print(time.strftime("%H:%M"), "|", lines[-1] if p.returncode == 0 and lines else f"{e} FAILED:\n{out[-2500:]}", flush=True)
            if p.returncode != 0:
                failed.append(e)
        if dev not in running and todo:
            e = todo.pop(0)
            running[dev] = (e, subprocess.Popen([sys.executable, "infer_ner.py", "--run", runs[e], "--out",
                                                 f"/kaggle/working/logits/{e}", "--device", dev],
                                                stdout=open(f"/kaggle/working/logits/{e}.out", "w"),
                                                stderr=subprocess.STDOUT, cwd="/kaggle/working"))
    time.sleep(3)
print("failed:", failed or "none")

## N10. Part C — train new experiments (only if `SESSION_EXPERIMENTS` is not empty)
Step 1: CPU smoke test of the experiments in `SMOKE_EXPERIMENTS` + checkpoint check. Step 2: training on the GPUs
(a distillation student waits for its teacher, a `train_on_all` run waits for its `epochs_from` run).

In [ ]:
# Part C, step 1: data / checkpoint checks + CPU smoke test of the experiments in SMOKE_EXPERIMENTS (tiny random
# models, a few documents; a distillation student's teacher is smoke-tested first automatically).
# Every experiment must end with exit code 0. Skipped when the list is empty.
import json, subprocess, sys
C = json.load(open("/kaggle/working/run_config_ner.json"))
smoke = []
for exp in SMOKE_EXPERIMENTS:
    t = C["experiments"][exp].get("teacher")
    smoke += [x for x in ([t] if t else []) + [exp] if x not in smoke]
for exp in smoke:
    r = subprocess.run([sys.executable, "train_ner.py", "--exp", exp, "--device", "cpu", "--smoke"],
                       capture_output=True, text=True, cwd="/kaggle/working")
    lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
    print(f"===== {exp}: exit code {r.returncode}")
    print("\n".join(lines[-6:] if r.returncode == 0 else lines[-25:]))
import ner_common as L, torch
for arch in sorted({C["experiments"][e]["arch"] for e in SESSION_EXPERIMENTS}):
    try:
        path = L.find_input(C["checkpoints"][arch])
        ck = torch.load(path, map_location="cpu", weights_only=False)
        print(f"checkpoint {arch:9s}: {path} | step {ck.get('step')} | val_loss {ck.get('val_loss', float('nan')):.3f}")
    except Exception as e:
        print(f"checkpoint {arch:9s}: MISSING -> {e}")

In [ ]:
# Part C, step 2: train SESSION_EXPERIMENTS — one experiment per GPU at a time; a distillation student waits for
# its teacher. Progress every 2 minutes. Each run also saves val/test token scores for the lab (N11).
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_ner.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending, running, finished = list(SESSION_EXPERIMENTS), {}, {}

def ready(exp):
    e = C["experiments"][exp]
    waits = [x for x in (e.get("teacher"), e.get("epochs_from")) if x]
    return all(t not in pending and t not in [r for r, _ in running.values()] for t in waits)

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][1].poll() is not None:
            exp, p = running.pop(dev); finished[exp] = p.returncode
            print(time.strftime("%H:%M"), f"| {exp} finished on {dev} with exit code {p.returncode}", flush=True)
        if dev not in running:
            nxt = next((e for e in pending if ready(e)), None)
            if nxt:
                pending.remove(nxt)
                out = open(f"/kaggle/working/{nxt}.out", "w")
                running[dev] = (nxt, subprocess.Popen([sys.executable, "train_ner.py", "--exp", nxt, "--device", dev],
                                                      stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt} on {dev}", flush=True)
    if time.time() - last_print > 120 and running:
        for dev, (exp, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{exp}.out").read().splitlines() if l.startswith(exp)]
            print(time.strftime("%H:%M"), "|", (lines[-1] if lines else f"{exp}: starting...")[:230], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", finished or "nothing trained this session")
for exp, code_ in finished.items():
    if code_ != 0:
        print(f"\n===== {exp} FAILED — last lines:\n" + open(f"/kaggle/working/{exp}.out").read()[-3000:])

## N11. The lab — decode, propagate and ensemble everything that has token scores
Per run: token / piece alignment × lenient / Viterbi × propagation thresholds; your `PROB_ENSEMBLES`; greedy ensembles
(one per alignment × decoding). A run trained with `align="piece"` is only decoded at piece level. Writes the
`TOP_N_ZIPS` best sets on validation + the best set of each member list; `index.csv` lists every set.

In [ ]:
# N11 — the lab (CPU, ~5-10 min): every run with token scores (Part B re-computed + Part C trained) is decoded
# token/piece x lenient/viterbi x each propagation threshold; your PROB_ENSEMBLES; greedy ensembles; plus Part A
# again (so runs trained this session are included). Writes zips + submissions/index.csv.
import json, importlib, ner_common as L, ner_lab
importlib.reload(L); importlib.reload(ner_lab)
C = json.load(open("/kaggle/working/run_config_ner.json"))
lab = ner_lab.Lab(C["n_val_docs"], C["val_fraction"])
lab.span_level(PROP_MIN_FRACS, VOTE_MEMBERS, RUN_DIR_OVERRIDES)
lab.load_scores()
lab.singles(PROP_MIN_FRACS)
lab.ensembles(PROB_ENSEMBLES, PROP_MIN_FRACS)
if GREEDY:
    lab.greedy(PROP_MIN_FRACS)
rows = lab.finish(TOP_N_ZIPS)
ner_lab.table(rows, 30)

## N12. Check the zips + confirm the lab's scores with the official evaluator

In [ ]:
# N12 — check every zip (format, test IDs, duplicates, labels) and confirm the lab's validation scores with the
# official evaluator on the 5 best written prediction sets.
import zipfile, json, glob, csv, subprocess, sys
test_ids = set(json.load(open("/kaggle/working/p2data/fold3-text.json")))
labels = {l.strip() for l in open("/kaggle/working/p2data/labels.txt") if l.strip()}
zips = sorted(glob.glob("/kaggle/working/submissions/*.zip")) + sorted(glob.glob("/kaggle/working/ner_runs/*/submission_*.zip"))
bad_zips = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    dup = sum(len(v) - len({tuple(x) for x in v}) for v in pred.values())
    bad = sum(1 for v in pred.values() for x in v if x[2] not in labels or not (0 <= x[0] < x[1]))
    ok = names == ["prediction.json"] and set(pred) == test_ids and dup == 0 and bad == 0
    bad_zips += not ok
    if not ok:
        print(f"PROBLEM {z} | docs {len(pred)} | duplicates {dup} | invalid {bad}")
print(f"{len(zips)} zips checked, {bad_zips} with problems")
V = "/kaggle/working/submissions/val_predictions"
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv")) if r["file"].endswith(".zip") and r["val_f1"]]
for r in rows[:5]:
    out = subprocess.run([sys.executable, "evaluate_phase2.py", "--truth", f"{V}/val_truth.json", "--prediction",
                          f"{V}/{r['name']}.json", "--labels", "/kaggle/working/p2data/labels.txt"],
                         capture_output=True, text=True)
    off = [l.split()[-1] for l in out.stdout.splitlines() if l.startswith("Strict Macro-F1")] or [out.stderr.strip()[-200:]]
    print(f"{r['name'][:70]:70s} lab {r['val_f1']} | official {off[0]}")

## N13. Error analysis of one prediction set (material for the report)

In [ ]:
# N13 — error analysis of one written prediction set (material for the report): error types + examples.
import json, csv, ner_common as L
NAME = None          # <- a name from submissions/index.csv; None = the best written one
V = "/kaggle/working/submissions/val_predictions"
if NAME is None:
    NAME = next(r["name"] for r in csv.DictReader(open("/kaggle/working/submissions/index.csv"))
                if r["file"].endswith(".zip") and r["val_f1"])
print("analysing:", NAME)
texts, gold, fold, test = L.load_data()
truth, pred = json.load(open(f"{V}/val_truth.json")), json.load(open(f"{V}/{NAME}.json"))
print("error types:", L.error_breakdown(truth, pred))
s = L.strict_scores(truth, pred)
print("per-label F1:", {l: round(v["F1"], 3) for l, v in s["per_label"].items()})
shown = 0
for d in truth:
    g = {(a, b, l) for a, b, l in truth[d]}; p = {(a, b, l) for a, b, l in pred[d]}
    for a, b, l in sorted(p - g):
        near = [x for x in g if x[0] < b and a < x[1]]
        if near and shown < 15:
            ga, gb, gl = near[0]
            print(f"  pred {l:9s} {texts[d][a:b]!r:45s} | gold {gl:9s} {texts[d][ga:gb]!r}")
            shown += 1